In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('../Dataset/final.csv')

In [5]:
df.drop("subject", axis=1, inplace=True)

In [6]:
df.columns

Index(['acc_x_mean', 'acc_x_std', 'acc_x_min', 'acc_x_max', 'acc_y_mean',
       'acc_y_std', 'acc_y_min', 'acc_y_max', 'acc_z_mean', 'acc_z_std',
       'acc_z_min', 'acc_z_max', 'ecg_mean', 'ecg_std', 'ecg_min', 'ecg_max',
       'emg_mean', 'emg_std', 'emg_min', 'emg_max', 'eda_mean', 'eda_std',
       'eda_min', 'eda_max', 'temp_mean', 'temp_std', 'temp_min', 'temp_max',
       'resp_mean', 'resp_std', 'resp_min', 'resp_max', 'net_acc_mean',
       'net_acc_std', 'net_acc_min', 'net_acc_max', 'ecg_peak_freq',
       'resp_peak_freq', 'temp_slope', 'eda_slope', 'label'],
      dtype='str')

In [10]:
accept_label = [1, 2, 3, 4]
df = df[df["label"].isin(accept_label)]
df["label"].value_counts()

label
1    581
4    387
2    326
3    180
Name: count, dtype: int64

In [11]:
nStr = [1, 3, 4]
def apply_target(label):
    if label == 2:
        return 1
    elif label in nStr:
        return 0
    else:
        return "unknown"

df["label"] = df["label"].apply(apply_target)

In [12]:
df["label"].value_counts()

label
0    1148
1     326
Name: count, dtype: int64

In [15]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

X_train = df.drop("label", axis=1)
y_train = df["label"]


kf = KFold(n_splits=10, shuffle=True, random_state=42)

scores = []

for train_index, val_index in kf.split(X_train):

    # Use iloc because it's a DataFrame
    X_tr = X_train.iloc[train_index]
    X_val = X_train.iloc[val_index]
    y_tr = y_train.iloc[train_index]
    y_val = y_train.iloc[val_index]

    

    model = XGBClassifier(
        n_estimators=30,
        max_depth=4,
        learning_rate=0.2,
        subsample=0.7,
        colsample_bytree=0.7,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        eval_metric="logloss",
        objective="binary:logistic",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_tr, y_tr), (X_val, y_val)],
        verbose=False
    )

    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)

    scores.append(acc)

print("Fold scores:", scores)
print("Mean accuracy:", sum(scores)/len(scores))

Fold scores: [0.9594594594594594, 0.9864864864864865, 0.9864864864864865, 0.9864864864864865, 0.9931972789115646, 0.9591836734693877, 0.9795918367346939, 0.9659863945578231, 0.9727891156462585, 0.9931972789115646]
Mean accuracy: 0.9782864497150211
